In [ ]:
# Import Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
from google.colab import drive


In [ ]:
# Mount Google Drive + Define file paths
drive.mount('/content/drive', force_remount=True)
DATA_DIR = '/content/drive/MyDrive/xg-ids/raw_datasets'
TRAIN_FILE = "nslkdd.txt"

Mounted at /content/drive


In [ ]:
# Column names for the NSL-KDD dataset
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty'
]

# Dictonary maps all the subcatagories to major catagories
subcategory_to_major = {
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS', 'mailbomb': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'named': 'R2L', 'xsnoop': 'R2L', 'xlock': 'R2L', 'sendmail': 'R2L', 'worm': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'httptunnel': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R', 'ps': 'U2R',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'normal': 'Normal'
}

# Load in the raw nslkdd dataset
nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/raw_datasets/nslkdd.txt', header=None, names=column_names)

In [ ]:
# Find all instances with samples with less than 20 instances
real_spy_sample = nslkdd[nslkdd['class'] == 'spy'].copy()
print(f"spy instances found: {len(real_spy_sample)}")

real_perl_sample = nslkdd[nslkdd['class'] == 'perl'].copy()
print(f"perl instances found: {len(real_perl_sample)}")

real_phf_sample = nslkdd[nslkdd['class'] == 'phf'].copy()
print(f"phf instances found: {len(real_phf_sample)}")

real_multihop_sample = nslkdd[nslkdd['class'] == 'multihop'].copy()
print(f"multihop instances found: {len(real_multihop_sample)}")

real_ftpwrite_sample = nslkdd[nslkdd['class'] == 'ftp_write'].copy()
print(f"ftp_write instances found: {len(real_ftpwrite_sample)}")

real_loadmodule_sample = nslkdd[nslkdd['class'] == 'loadmodule'].copy()
print(f"loadmodule instances found: {len(real_loadmodule_sample)}")

real_rootkit_sample = nslkdd[nslkdd['class'] == 'rootkit'].copy()
print(f"rootkit instances found: {len(real_rootkit_sample)}")

real_imap_sample = nslkdd[nslkdd['class'] == 'imap'].copy()
print(f"imap instances found: {len(real_imap_sample)}")

real_land_sample = nslkdd[nslkdd['class'] == 'land'].copy()
print(f"land instances found: {len(real_land_sample)}")

micro_samples = pd.concat([
    real_spy_sample,
    real_perl_sample,
    real_phf_sample,
    real_multihop_sample,
    real_ftpwrite_sample,
    real_loadmodule_sample,
    real_rootkit_sample,
    real_imap_sample,
    real_land_sample
], ignore_index=True)

# Remove micro samples from the main dataset before splitting
nslkdd_clean = nslkdd[~nslkdd['class'].isin(micro_samples['class'].unique())].copy()

spy instances found: 2
perl instances found: 3
phf instances found: 4
multihop instances found: 7
ftp_write instances found: 8
loadmodule instances found: 9
rootkit instances found: 10
imap instances found: 11
land instances found: 18


Split Original nslkdd into 90/10 Train/Validate sets

In [ ]:
# Split the dataset 90(train)/10(validate)
df_train, df_validate = train_test_split(
    nslkdd_clean,
    test_size=0.10,
    random_state=42,
    stratify=nslkdd_clean['class']
)

# Insert instances of all micro samples into both train and validate set
df_validate = pd.concat([df_validate, micro_samples], ignore_index=True)
df_train = pd.concat([df_train, micro_samples], ignore_index=True)

# Save the train dataset
df_train.to_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_TRAIN.txt', index=False)

# Save the validate dataset
df_validate.to_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_VALIDATE.txt', index=False)

In [ ]:
print(micro_samples['class'].value_counts())

class
land          18
imap          11
rootkit       10
loadmodule     9
ftp_write      8
multihop       7
phf            4
perl           3
spy            2
Name: count, dtype: int64


#~~~~~ VALIDATE DATASET PROCESSING ~~~~~

In [ ]:

# Load in the validate dataset (with headers)
validate_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_VALIDATE.txt')

# Displays preview of the validate dataset
print(f"Total instances: {validate_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label, and 1 difficulty ranking): {validate_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset before processing:")
validate_nslkdd.head()

Total instances: 12663
Total Columns (41 features, 1 class label, and 1 difficulty ranking): 43
Preview of the first 5 instances of the NSL-KDD Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,kshell,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,2,0.0,0.0,1.0,1.0,0.02,0.07,0.0,255,2,0.01,0.07,0.00,0.00,0.0,0.0,1.0,1.0,neptune,20
1,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,15,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,25,1.00,0.00,1.00,0.56,0.0,0.0,0.0,0.0,ipsweep,16
2,0,tcp,http,SF,331,479,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.0,0.0,0.0,0.0,1.00,0.00,0.0,68,255,1.00,0.00,0.01,0.01,0.0,0.0,0.0,0.0,normal,21
3,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,22,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,86,1.00,0.00,1.00,0.51,0.0,0.0,0.0,0.0,ipsweep,17
4,0,tcp,rje,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,210,5,1.0,1.0,0.0,0.0,0.02,0.07,0.0,255,5,0.02,0.08,0.00,0.00,1.0,1.0,0.0,0.0,neptune,21


In [ ]:
# Dropping difficulty column (last column) from validate dataset
validate_nslkdd = validate_nslkdd.drop('difficulty', axis=1)
# Displays preview of the validate dataset after dropping difficulty column
print(f"Total Columns (41 features and 1 class label: {validate_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:")
validate_nslkdd.head()

Total Columns (41 features and 1 class label: 42
Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,kshell,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,2,0.0,0.0,1.0,1.0,0.02,0.07,0.0,255,2,0.01,0.07,0.00,0.00,0.0,0.0,1.0,1.0,neptune
1,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,15,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,25,1.00,0.00,1.00,0.56,0.0,0.0,0.0,0.0,ipsweep
2,0,tcp,http,SF,331,479,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.0,0.0,0.0,0.0,1.00,0.00,0.0,68,255,1.00,0.00,0.01,0.01,0.0,0.0,0.0,0.0,normal
3,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,22,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,86,1.00,0.00,1.00,0.51,0.0,0.0,0.0,0.0,ipsweep
4,0,tcp,rje,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,210,5,1.0,1.0,0.0,0.0,0.02,0.07,0.0,255,5,0.02,0.08,0.00,0.00,1.0,1.0,0.0,0.0,neptune


In [ ]:
# Rename subcategories to major categories
validate_nslkdd['class'] = validate_nslkdd['class'].map(subcategory_to_major)


# Displays preview of the validate dataset after subcatagory conversion
unique_classes = validate_nslkdd['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(validate_nslkdd['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: ")
validate_nslkdd.head()



5 unique classes:
  DoS: 4610
  Normal: 6735
  Probe: 1165
  R2L: 128
  U2R: 25
Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,kshell,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,2,0.0,0.0,1.0,1.0,0.02,0.07,0.0,255,2,0.01,0.07,0.00,0.00,0.0,0.0,1.0,1.0,DoS
1,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,15,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,25,1.00,0.00,1.00,0.56,0.0,0.0,0.0,0.0,Probe
2,0,tcp,http,SF,331,479,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.0,0.0,0.0,0.0,1.00,0.00,0.0,68,255,1.00,0.00,0.01,0.01,0.0,0.0,0.0,0.0,Normal
3,0,icmp,eco_i,SF,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,22,0.0,0.0,0.0,0.0,1.00,0.00,1.0,1,86,1.00,0.00,1.00,0.51,0.0,0.0,0.0,0.0,Probe
4,0,tcp,rje,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,210,5,1.0,1.0,0.0,0.0,0.02,0.07,0.0,255,5,0.02,0.08,0.00,0.00,1.0,1.0,0.0,0.0,DoS


In [ ]:
# Export the validate dataset after subcatagory conversion
validate_nslkdd.to_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_VALIDATE.txt', index=False)
print(f"Saved. Total instances: {validate_nslkdd.shape[0]}")

Saved. Total instances: 12663


#~~~~~ TRAIN DATASET PROCESSING ~~~~~

---



In [ ]:
# Load in the train dataset + apply column/feature headers
train_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_TRAIN.txt')

pre_smote_subcategory_quantity = dict(train_nslkdd['class'].value_counts())

# Displays preview of the train dataset
print(f"Total instances: {train_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label, and 1 difficulty ranking): {train_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset before processing:")
train_nslkdd.head()

Total instances: 113382
Total Columns (41 features, 1 class label, and 1 difficulty ranking): 43
Preview of the first 5 instances of the NSL-KDD Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,udp,domain_u,SF,29,29,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,3,0.0,0.0,0.0,0.0,1.00,0.00,0.67,255,255,1.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,normal,21
1,0,tcp,login,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,7,1.0,1.0,0.0,0.0,0.88,0.25,0.00,255,7,0.03,0.07,0.00,0.00,1.0,1.0,0.0,0.0,neptune,18
2,0,tcp,http,SF,334,13361,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,4,6,0.0,0.0,0.0,0.0,1.00,0.00,0.50,22,255,1.00,0.00,0.05,0.04,0.0,0.0,0.0,0.0,normal,21
3,0,tcp,ftp,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,266,7,1.0,1.0,0.0,0.0,0.03,0.06,0.00,255,7,0.03,0.08,0.00,0.00,1.0,1.0,0.0,0.0,neptune,21
4,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,150,25,1.0,1.0,0.0,0.0,0.17,0.05,0.00,255,6,0.02,0.07,0.00,0.00,1.0,1.0,0.0,0.0,neptune,20


In [ ]:
# Dropping difficulty column (last column) from train dataset
train_nslkdd = train_nslkdd.drop('difficulty', axis=1)

# Displays preview of the train dataset after dropping difficulty column
print(f"Total Columns (41 features and 1 class label: {train_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the Train Dataset after dropping difficulty column:")
train_nslkdd.head()

Total Columns (41 features and 1 class label: 42
Preview of the first 5 instances of the Train Dataset after dropping difficulty column:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,udp,domain_u,SF,29,29,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,3,0.0,0.0,0.0,0.0,1.00,0.00,0.67,255,255,1.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,normal
1,0,tcp,login,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8,7,1.0,1.0,0.0,0.0,0.88,0.25,0.00,255,7,0.03,0.07,0.00,0.00,1.0,1.0,0.0,0.0,neptune
2,0,tcp,http,SF,334,13361,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,4,6,0.0,0.0,0.0,0.0,1.00,0.00,0.50,22,255,1.00,0.00,0.05,0.04,0.0,0.0,0.0,0.0,normal
3,0,tcp,ftp,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,266,7,1.0,1.0,0.0,0.0,0.03,0.06,0.00,255,7,0.03,0.08,0.00,0.00,1.0,1.0,0.0,0.0,neptune
4,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,150,25,1.0,1.0,0.0,0.0,0.17,0.05,0.00,255,6,0.02,0.07,0.00,0.00,1.0,1.0,0.0,0.0,neptune


In [ ]:
# Identify and track columns containing string data types (excluding the class label)
categorical_cols = []

for col in train_nslkdd.columns:
    if train_nslkdd[col].dtype.name in ('object', 'string', 'str') and col != 'class':
        categorical_cols.append(col)

print(f"Categorical columns found: {categorical_cols}")
print(f"\nUnique values per column:")
for col in categorical_cols:
    print(f"  {col}: {train_nslkdd[col].nunique()}")

Categorical columns found: ['protocol_type', 'service', 'flag']

Unique values per column:
  protocol_type: 3
  service: 70
  flag: 11


In [ ]:
# Use pandas to convert categoricals + save the mappings
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_nslkdd[col] = le.fit_transform(train_nslkdd[col])
    label_encoders[col] = le

# Encode the class label separately
le_class = LabelEncoder()
train_nslkdd['class'] = le_class.fit_transform(train_nslkdd['class'])
print("Categorical columns encoded.")

# Map class names to their encoded integers for sampling_strategy reference
class_map = dict(zip(le_class.classes_, le_class.transform(le_class.classes_)))
print("Class encoding map:")
print(class_map)

Categorical columns encoded.
Class encoding map:
{'back': np.int64(0), 'buffer_overflow': np.int64(1), 'ftp_write': np.int64(2), 'guess_passwd': np.int64(3), 'imap': np.int64(4), 'ipsweep': np.int64(5), 'land': np.int64(6), 'loadmodule': np.int64(7), 'multihop': np.int64(8), 'neptune': np.int64(9), 'nmap': np.int64(10), 'normal': np.int64(11), 'perl': np.int64(12), 'phf': np.int64(13), 'pod': np.int64(14), 'portsweep': np.int64(15), 'rootkit': np.int64(16), 'satan': np.int64(17), 'smurf': np.int64(18), 'spy': np.int64(19), 'teardrop': np.int64(20), 'warezclient': np.int64(21), 'warezmaster': np.int64(22)}


In [ ]:
# Initialize SMOTE
X = train_nslkdd.drop('class', axis=1)
y = train_nslkdd['class']

In [ ]:
#SMOTE 1: Targeting Spy subcatagory (neighbor=1) + Generate: 5
spy_encoded = class_map['spy']
current_count = (y == spy_encoded).sum()
smote1 = SMOTE(random_state=42, k_neighbors=1, sampling_strategy={spy_encoded: current_count + 5})
X_res, y_res = smote1.fit_resample(X, y)
print(f"SMOTE 1 done. spy: {current_count} → {current_count + 5}")

#SMOTE 2: Targeting perl subcatagory (neighbor=2) + Generate: 5
perl_encoded = class_map['perl']
current_count = (y_res == perl_encoded).sum()
smote2 = SMOTE(random_state=42, k_neighbors=2, sampling_strategy={perl_encoded: current_count + 5})
X_res, y_res = smote2.fit_resample(X_res, y_res)
print(f"SMOTE 2 done. perl: {current_count} → {current_count + 5}")

#SMOTE 3: Targeting phf subcatagory (neighbor=3) + Generate: 7
phf_encoded = class_map['phf']
current_count = (y_res == phf_encoded).sum()
smote3 = SMOTE(random_state=42, k_neighbors=3, sampling_strategy={phf_encoded: current_count + 7})
X_res, y_res = smote3.fit_resample(X_res, y_res)
print(f"SMOTE 3 done. phf: {current_count} → {current_count + 7}")

#SMOTE 4: Targeting multihop subcatagory (neighbor=5) + Generate: 15
multihop_encoded = class_map['multihop']
current_count = (y_res == multihop_encoded).sum()
smote4 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={multihop_encoded: current_count + 15})
X_res, y_res = smote4.fit_resample(X_res, y_res)
print(f"SMOTE 4 done. multihop: {current_count} → {current_count + 15}")

#SMOTE 5: Targeting ftp_write subcatagory (neighbor=5)+ Generate: 18
ftp_encoded = class_map['ftp_write']
current_count = (y_res == ftp_encoded).sum()
smote5 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={ftp_encoded: current_count + 18})
X_res, y_res = smote5.fit_resample(X_res, y_res)
print(f"SMOTE 5 done. ftp_write: {current_count} → {current_count + 18}")

#SMOTE 6: Targeting loadmodule subcatagory (neighbor=5) + Generate: 20
loadmodule_encoded = class_map['loadmodule']
current_count = (y_res == loadmodule_encoded).sum()
smote6 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={loadmodule_encoded: current_count + 20})
X_res, y_res = smote6.fit_resample(X_res, y_res)
print(f"SMOTE 6 done. loadmodule: {current_count} → {current_count + 20}")

#SMOTE 7: Targeting rootkit subcatagory (neighbor=5) + Generate: 25
rootkit_encoded = class_map['rootkit']
current_count = (y_res == rootkit_encoded).sum()
smote7 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={rootkit_encoded: current_count + 25})
X_res, y_res = smote7.fit_resample(X_res, y_res)
print(f"SMOTE 7 done. rootkit: {current_count} → {current_count + 25}")

#SMOTE 8: Targeting imap subcatagory (neighbor=5) + Generate: 28
imap_encoded = class_map['imap']
current_count = (y_res == imap_encoded).sum()
smote8 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={imap_encoded: current_count + 28})
X_res, y_res = smote8.fit_resample(X_res, y_res)
print(f"SMOTE 8 done. imap: {current_count} → {current_count + 28}")

#SMOTE 9: Targeting land subcatagory (neighbor=5) + Generate: 48
land_encoded = class_map['land']
current_count = (y_res == land_encoded).sum()
smote9 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={land_encoded: current_count + 48})
X_res, y_res = smote9.fit_resample(X_res, y_res)
print(f"SMOTE 9 done. land: {current_count} → {current_count + 48}")

#SMOTE 10: Targeting warezmaster subcatagory(neighbor=5) + Generate: 50
warezmaster_encoded = class_map['warezmaster']
current_count = (y_res == warezmaster_encoded).sum()
smote10 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={warezmaster_encoded: current_count + 50})
X_res, y_res = smote10.fit_resample(X_res, y_res)
print(f"SMOTE 10 done. warezmaster: {current_count} → {current_count + 50}")

#SMOTE 11: Targeting buffer_overflow subcatagory(neighbor=5) + Generate: 70
buffer_overflow_encoded = class_map['buffer_overflow']
current_count = (y_res == buffer_overflow_encoded).sum()
smote11 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={buffer_overflow_encoded: current_count + 70})
X_res, y_res = smote11.fit_resample(X_res, y_res)
print(f"SMOTE 11 done. buffer_overflow: {current_count} → {current_count + 70}")

#SMOTE 12: Targeting guess_passwd subcatagory (neighbor=5) + Generate: 100
guess_passwd_encoded = class_map['guess_passwd']
current_count = (y_res == guess_passwd_encoded).sum()
smote12 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={guess_passwd_encoded: current_count + 100})
X_res, y_res = smote12.fit_resample(X_res, y_res)
print(f"SMOTE 12 done. guess_passwd: {current_count} → {current_count + 100}")

#SMOTE 13: Targeting pod subcatagory (neighbor=5) + Generate: 200
pod_encoded = class_map['pod']
current_count = (y_res == pod_encoded).sum()
smote13 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={pod_encoded: current_count + 200})
X_res, y_res = smote13.fit_resample(X_res, y_res)
print(f"SMOTE 13 done. pod: {current_count} → {current_count + 200}")

#SMOTE 14: Targeting warezclient subcatagory (neighbor=5) + Generate: 200
warezclient_encoded = class_map['warezclient']
current_count = (y_res == warezclient_encoded).sum()
smote14 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={warezclient_encoded: current_count + 200})
X_res, y_res = smote14.fit_resample(X_res, y_res)
print(f"SMOTE 14 done. warezclient: {current_count} → {current_count + 200}")

#SMOTE 15: Targeting teardrop subcatagory (neighbor=5) + Generate: 200
teardrop_encoded = class_map['teardrop']
current_count = (y_res == teardrop_encoded).sum()
smote15 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={teardrop_encoded: current_count + 200})
X_res, y_res = smote15.fit_resample(X_res, y_res)
print(f"SMOTE 15 done. teardrop: {current_count} → {current_count + 200}")

#SMOTE 16: Targeting back subcatagory (neighbor=5) + Generate: 200
back_encoded = class_map['back']
current_count = (y_res == back_encoded).sum()
smote16 = SMOTE(random_state=42, k_neighbors=5, sampling_strategy={back_encoded: current_count + 200})
X_res, y_res = smote16.fit_resample(X_res, y_res)
print(f"SMOTE 16 done. back: {current_count} → {current_count + 200}")

SMOTE 1 done. spy: 2 → 7
SMOTE 2 done. perl: 3 → 8
SMOTE 3 done. phf: 4 → 11
SMOTE 4 done. multihop: 7 → 22
SMOTE 5 done. ftp_write: 8 → 26
SMOTE 6 done. loadmodule: 9 → 29
SMOTE 7 done. rootkit: 10 → 35
SMOTE 8 done. imap: 11 → 39
SMOTE 9 done. land: 18 → 66
SMOTE 10 done. warezmaster: 18 → 68
SMOTE 11 done. buffer_overflow: 27 → 97
SMOTE 12 done. guess_passwd: 48 → 148
SMOTE 13 done. pod: 181 → 381
SMOTE 14 done. warezclient: 801 → 1001
SMOTE 15 done. teardrop: 803 → 1003
SMOTE 16 done. back: 860 → 1060


In [ ]:
# Decode the class labels in the train.txt file back to their original string values using the saved LabelEncoder mappings.
df_final = pd.DataFrame(X_res, columns=X.columns)
df_final['class'] = y_res

for col in categorical_cols:
    df_final[col] = label_encoders[col].inverse_transform(df_final[col].astype(int))
df_final['class'] = le_class.inverse_transform(df_final['class'])

# Remove original micro samples from the train dataset after SMOTE
micro_class_counts = micro_samples['class'].value_counts().to_dict()

cleaned_parts = []
for cls, count in micro_class_counts.items():
    cls_rows = df_final[df_final['class'] == cls]
    cleaned_parts.append(cls_rows.iloc[count:])

non_micro = df_final[~df_final['class'].isin(micro_class_counts.keys())]
df_final = pd.concat([non_micro] + cleaned_parts, ignore_index=True)

In [ ]:
# POST-SMOTE Sanity Check

# Count quantity of each subcategory
decoded_classes = le_class.inverse_transform(y_res)
post_smote_subcategory_quantity = dict(df_final['class'].value_counts())

# Check the train.txt file for NaN/null values
nan_count = pd.DataFrame(X_res).isnull().sum().sum()

# Feature column count
feature_column_count = X_res.shape[1]

# Display results table
print(f"Feature column count: {feature_column_count}")
print(f"NaN/Null values found: {nan_count}\n")

print(f"{'Subcategory':<20} {'Original':>10} {'Post-SMOTE':>12} {'Delta':>8}")
print("-" * 52)
for cls in sorted(pre_smote_subcategory_quantity.keys()):
    original = pre_smote_subcategory_quantity[cls]
    post = post_smote_subcategory_quantity.get(cls, 0)
    delta = post - original
    print(f"{cls:<20} {original:>10} {post:>12} {delta:>+8}")

total_original = sum(pre_smote_subcategory_quantity.values())
total_post = total_post = len(df_final)
print("-" * 52)
print(f"{'TOTAL':<20} {total_original:>10} {total_post:>12} {total_post - total_original:>+8}")

Feature column count: 41
NaN/Null values found: 0

Subcategory            Original   Post-SMOTE    Delta
----------------------------------------------------
back                        860         1060     +200
buffer_overflow              27           97      +70
ftp_write                     8           18      +10
guess_passwd                 48          148     +100
imap                         11           28      +17
ipsweep                    3239         3239       +0
land                         18           48      +30
loadmodule                    9           20      +11
multihop                      7           15       +8
neptune                   37092        37092       +0
nmap                       1344         1344       +0
normal                    60608        60608       +0
perl                          3            5       +2
phf                           4            7       +3
pod                         181          381     +200
portsweep                  2638 

In [ ]:
print(train_nslkdd['class'].unique())

[11  9 10 18 14  5 17 15 20  0 21  3  1 22 19 12 13  8  2  7 16  4  6]


In [ ]:
# TODO: Balance Training Dataset
sample_neptune = df_final[df_final['class'] == 'neptune'].sample(n=9127, random_state=42)
sample_normal = df_final[df_final['class'] == 'normal'].sample(n=14000, random_state=42)
print(f"neptune samples: {len(sample_neptune)}")
print(f"normal samples: {len(sample_normal)}")

neptune samples: 9127
normal samples: 14000


In [ ]:
# Create a separate variable called “nslkdd_exclude_normal_neptune”. Copies original dataset but excludes all instances of “normal” and all instances of “neptune”
nslkdd_exclude_normal_neptune = df_final[(df_final['class'] != 'normal') & (df_final['class'] != 'neptune')]
print(f"Excluded dataset instances: {nslkdd_exclude_normal_neptune.shape[0]}")

Excluded dataset instances: 16801


In [ ]:
# Create a new dataset that concatenates “sample_neptune” + “sample_normal” + “nslkdd_exclude_normal_neptune”
nslkdd_balanced = pd.concat([sample_neptune, sample_normal, nslkdd_exclude_normal_neptune]).reset_index(drop=True)
print(f"Total instances after balancing: {nslkdd_balanced.shape[0]}")

Total instances after balancing: 39928


In [ ]:
# Rename subcategories to major categories
nslkdd_balanced['class'] = nslkdd_balanced['class'].map(subcategory_to_major)

unique_classes = nslkdd_balanced['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(nslkdd_balanced['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Train Dataset after subcategory conversion: ")
nslkdd_balanced.head()

5 unique classes:
  DoS: 14000
  Normal: 14000
  Probe: 10491
  R2L: 1290
  U2R: 147
Preview of the first 5 instances of the Train Dataset after subcategory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,domain,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,297,12,1.0,1.0,0.0,0.0,0.04,0.05,0.0,255,12,0.05,0.05,0.0,0.0,1.0,1.0,0.0,0.0,DoS
1,0,tcp,printer,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9,5,1.0,1.0,0.0,0.0,0.56,0.33,0.0,255,5,0.02,0.07,0.0,0.0,1.0,1.0,0.0,0.0,DoS
2,0,tcp,systat,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,116,2,1.0,1.0,0.0,0.0,0.02,0.07,0.0,255,2,0.01,0.07,0.0,0.0,1.0,1.0,0.0,0.0,DoS
3,0,tcp,netbios_ns,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,247,20,0.0,0.0,1.0,1.0,0.08,0.06,0.0,255,20,0.08,0.07,0.0,0.0,0.0,0.0,1.0,1.0,DoS
4,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,228,2,1.0,1.0,0.0,0.0,0.01,0.06,0.0,255,2,0.01,0.06,0.0,0.0,1.0,1.0,0.0,0.0,DoS


In [ ]:
nslkdd_balanced.to_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_TRAIN.txt', index=False)
print(f"Export complete. Total instances: {nslkdd_balanced.shape[0]}")

Export complete. Total instances: 39928


#~~~~~ TEST DATASET PROCESSING ~~~~~


In [ ]:
# Load in the Test dataset + Apply Column Headers
test_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/raw_datasets/KDDTest.txt', header=None, names=column_names)

# Displays preview of the validate dataset
print(f"Total instances: {test_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label, and 1 difficulty ranking): {test_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Test Dataset before processing:")
test_nslkdd.head()

Total instances: 22544
Total Columns (41 features, 1 class label, and 1 difficulty ranking): 43
Preview of the first 5 instances of the NSL-KDD Test Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [ ]:
# Dropping difficulty column (last column) from test dataset
test_nslkdd = test_nslkdd.drop('difficulty', axis=1)
# Displays preview of the validate dataset after dropping difficulty column
print(f"Total Columns (41 features and 1 class label: {test_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:")
test_nslkdd.head()

Total Columns (41 features and 1 class label: 42
Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan


In [ ]:
# Rename subcategories to major categories
test_nslkdd['class'] = test_nslkdd['class'].map(subcategory_to_major)


# Displays preview of the test dataset after subcatagory conversion
unique_classes = test_nslkdd['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(test_nslkdd['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: ")
test_nslkdd.head()

5 unique classes:
  DoS: 7458
  Normal: 9711
  Probe: 2421
  R2L: 2754
  U2R: 200
Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,DoS
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,DoS
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,Normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,Probe
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,Probe


In [ ]:
# Export the test dataset after subcatagory conversion
test_nslkdd.to_csv('/content/drive/MyDrive/xg-ids/5class/datasets/5class_TEST.txt', index=False)
print(f"Saved. Total instances: {test_nslkdd.shape[0]}")

Saved. Total instances: 22544
